In [0]:
from pyspark.sql.functions import col, avg, row_number, desc, round as _round, when, lit, concat_ws
from pyspark.sql.window import Window

# Load tables
ratings = spark.table("workspace.default.ratings")
users = spark.table("workspace.default.users")
movies = spark.table("workspace.default.movie_genre_silver")
occupations = spark.table("workspace.default.occupations")

# Cast string columns to proper types
ratings_typed = ratings.select(
    col("userId"),
    col("movieId"),
    col("rating").cast("double").alias("rating"),
    col("timestamp").cast("long").alias("timestamp")
)

users_typed = users.select(
    col("userId"),
    col("gender"),
    col("age").cast("int").alias("age"),
    col("occupation").cast("int").alias("occupation_id"),
    col("zip")
)

# Join ratings + users + occupations + movies
joined = (
    ratings_typed
    .join(users_typed, on="userId", how="inner")
    .join(occupations, on="occupation_id", how="left")
    .join(movies.withColumn("genres", concat_ws("|",
        when(col("genre_action"), lit("Action")),
        when(col("genre_adventure"), lit("Adventure")),
        when(col("genre_animation"), lit("Animation")),
        when(col("genre_children's"), lit("Children's")),
        when(col("genre_comedy"), lit("Comedy")),
        when(col("genre_crime"), lit("Crime")),
        when(col("genre_documentary"), lit("Documentary")),
        when(col("genre_drama"), lit("Drama")),
        when(col("genre_fantasy"), lit("Fantasy")),
        when(col("genre_film_noir"), lit("Film-Noir")),
        when(col("genre_horror"), lit("Horror")),
        when(col("genre_musical"), lit("Musical")),
        when(col("genre_mystery"), lit("Mystery")),
        when(col("genre_romance"), lit("Romance")),
        when(col("genre_sci_fi"), lit("Sci-Fi")),
        when(col("genre_thriller"), lit("Thriller")),
        when(col("genre_war"), lit("War")),
        when(col("genre_western"), lit("Western")),
    )).select("movieId", "title", "genres"), on="movieId", how="inner")
)

# 1. Average rating and average age per movie
movie_stats = (
    joined.groupBy("movieId", "title", "genres")
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        _round(avg("age"), 1).alias("avg_age"),
    )
)

# 2. Occupation that rated each movie the highest (by avg rating)
occ_movie = (
    joined.groupBy("movieId", "occupation_name")
    .agg(avg("rating").alias("occ_avg_rating"))
)

w = Window.partitionBy("movieId").orderBy(desc("occ_avg_rating"))
top_occupation = (
    occ_movie
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        "movieId",
        col("occupation_name").alias("top_occupation"),
        _round(col("occ_avg_rating"), 2).alias("top_occupation_avg_rating")
    )
)

# Final dataframe: one row per movie
movie_final = movie_stats.join(top_occupation, on="movieId", how="left").orderBy(desc("avg_rating"))

display(movie_final)

In [0]:
movie_final.write.mode("overwrite").saveAsTable("workspace.default.movie_stats_gold")
print("Saved to workspace.default.movie_stats_gold")